# Augment data

## Notes:

Binary variables:
- paraphrased
- misspelled
- capitalization_change
- punctuation_change

Each change is a new row. 

In [43]:
# augly + dependencies
!conda install -c conda-forge python-magic -y
!pip install augly[text] -q

# dependencies for OpenAI LLM call
!pip install openai python-dotenv -q

Retrieving notices: done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.7.0
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [44]:
# Import data

import pandas as pd
import augly.text as txtaugs

# MultiRC With Reference Answer and GPT5 4-point scores:
multirc_path = "../../data/multirc-data-w-gpt5-scores-v3.csv"

df = pd.read_csv(multirc_path, index_col=0)

# Drop rows with NaN responses — augly functions expect strings, not floats
df = df.dropna(subset=['response'])
print(f"Rows after dropping NaN responses: {len(df)}")

# Finding unique key - it appears to be provided_split & answer_id combined.
print(f"Rows with duplicate provided_split and answer_id: {df.duplicated(subset=['provided_split', 'answer_id']).sum()}")

Rows after dropping NaN responses: 32076
Rows with duplicate provided_split and answer_id: 0


## Patterns & Partitions

Re-use patterns from `5b-explore-training-data.ipynb` to create boolean masks that determine which rows each augmentation applies to.

- Date format --> rows whose response matches a date pattern
- Numeric to words --> rows whose response is digit-only or digit+unit
- Typos / caps / punct --> **all rows** (surface-level, semantically neutral)
- Shorter paraphrase --> `gpt5_score == 4`
- Longer parapharse --> `gpt5_score == 2`

The number of each pattern is printed out below the code cell.

In [45]:
import re
import random
from dateutil import parser as dateutil_parser

# --- Patterns ---
DATE_PATTERN = (
    r'\b(?:January|February|March|April|May|June|July|August|September|'
    r'October|November|December)\s+\d{1,2}\b'
)
DIGIT_ONLY_PATTERN = r'^\d+$'
DIGIT_WITH_UNIT_PATTERN = r'^\d+\s+\w+'

# --- Masks ---
date_mask       = df['response'].str.contains(DATE_PATTERN, case=False, na=False, regex=True)
digit_only_mask = df['response'].str.contains(DIGIT_ONLY_PATTERN, na=False, regex=True)
digit_unit_mask = df['response'].str.contains(DIGIT_WITH_UNIT_PATTERN, na=False, regex=True)
numeric_mask    = digit_only_mask | digit_unit_mask
score4_mask     = df['gpt5_score'] == 4
score2_mask     = df['gpt5_score'] == 2  

# Statistics
print(f"Date rows:              {date_mask.sum():>6}")
print(f"Numeric rows:           {numeric_mask.sum():>6}")
print(f"Score = 4 rows:         {score4_mask.sum():>6}")
print(f"Score = 2 rows:         {score2_mask.sum():>6}")
print(f"All rows (surface aug): {len(df):>6}")


Date rows:                  95
Numeric rows:             1678
Score = 4 rows:           1554
Score = 2 rows:           5024
All rows (surface aug):  32076


### Augmentation Function Definitions

In [46]:
# Binary flag columns — one per augmentation type
AUG_COLS = [
    'misspelled',
    'paraphrased',
    'capitalization_change',
    'punctuation_change',
    'date_reformatted',
    'numeric_words',
]

def _set_aug_flags(df_subset, active_col):
    """Set all AUG_COLS to 0, then set active_col to 1."""
    for col in AUG_COLS:
        df_subset[col] = 0
    df_subset[active_col] = 1
    return df_subset


# --- Date format augmentation ---
def augment_date(date_str, n=3):
    dt = dateutil_parser.parse(date_str)

    formats = [
        "%m/%d/%Y",      # 10/17/2026
        "%m/%d/%y",      # 10/17/26
        "%m/%d",         # 10/17
        "%B %d, %Y",     # October 17, 2026
        "%B %d",         # October 17
        "%b. %d, %Y",    # Oct. 17, 2026
        "%b. %d",        # Oct. 17
        "%b %d, %Y",     # Oct 17, 2026
        "%d %B %Y",      # 17 October 2026
        "%Y-%m-%d",      # 2026-10-17
    ]

    def ordinal(day):
        suffix = {1: "st", 2: "nd", 3: "rd"}.get(
            day % 10 * (day % 100 not in [11, 12, 13]), "th"
        )
        return f"{day}{suffix}"

    def format_date(dt, fmt):
        return dt.strftime(fmt).lstrip("0").replace("/0", "/").replace(" 0", " ")

    ordinal_formats = [
        f"{dt.strftime('%B')} {ordinal(dt.day)}, {dt.year}",
        f"{dt.strftime('%B')} {ordinal(dt.day)}",
        f"{dt.strftime('%b')}. {ordinal(dt.day)}, {dt.year}",
    ]

    results = [format_date(dt, f) for f in formats] + ordinal_formats
    return random.sample(results, min(n, len(results)))


# --- Numeric → spelled-out ---
def num_to_words(n):
    ones = [
        'zero', 'one', 'two', 'three', 'four', 'five', 'six', 'seven',
        'eight', 'nine', 'ten', 'eleven', 'twelve', 'thirteen', 'fourteen',
        'fifteen', 'sixteen', 'seventeen', 'eighteen', 'nineteen',
    ]
    tens_w = ['', '', 'twenty', 'thirty', 'forty', 'fifty', 'sixty', 'seventy', 'eighty', 'ninety']
    n = int(n)
    if n < 20:
        return ones[n]
    if n < 100:
        rest = ones[n % 10] if n % 10 else ''
        return tens_w[n // 10] + ('-' + rest if rest else '')
    return str(n)  # skip conversion for large numbers (e.g. years)

def augment_numeric(text):
    """Replace a leading digit sequence with its spelled-out equivalent."""
    match = re.match(r'^(\d+)(.*)', str(text))
    if not match:
        return text
    num_str, rest = match.groups()
    num = int(num_str)
    if num > 99:  # don't convert years or large values
        return text
    return num_to_words(num).capitalize() + rest


# --- Surface augmentations (all rows) ---
def augment_typos(texts):
    # Generate both keyboard (adjacent key) and common misspelling variants,
    # then randomly pick one per text for a natural mix across the dataset.
    keyboard_results = txtaugs.simulate_typos(texts=texts, typo_type="keyboard")
    misspelling_results = txtaugs.simulate_typos(texts=texts, typo_type="misspelling")
    return [
        random.choice([k, m])
        for k, m in zip(keyboard_results, misspelling_results)
    ]

def augment_capitalization(texts):
    results = []
    for t in texts:
        choice = random.choice(['lower', 'upper', 'title'])
        if choice == 'lower':
            results.append(t.lower())
        elif choice == 'upper':
            results.append(t.upper())
        else:
            results.append(t.title())
    return results

def augment_punctuation(texts):
    return txtaugs.insert_punctuation_chars(
        texts=texts,
        granularity="all",
        cadence=5.0,
        vary_chars=True,
    )

### LLM Paraphrase Function Definitions

- `score=4` rows → shorter paraphrases (reduces length bias for high-scoring answers)
- `score=2` rows → longer paraphrases (tests that verbosity alone doesn't raise scores)

All other columns — including `gpt5_score` — are copied from the original row unchanged.

In [47]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from tqdm import tqdm

load_dotenv("../.env")
client = OpenAI()

PARAPHRASE_PROMPTS = {
    'shorter': (
        "Rewrite the student response more concisely. "
        "Use fewer words but preserve the meaning exactly. "
        "Return only the rewritten response, no explanation."
    ),
    'longer': (
        "Rewrite the student response with more elaboration and detail. "
        "Preserve the original meaning exactly — do not add new correct information. "
        "Return only the rewritten response, no explanation."
    ),
}

def paraphrase_response(response, passage, question, answer, style='shorter'):
    prompt = (
        f"Passage: {passage[:500]}\n"
        f"Question: {question}\n"
        f"Reference answer: {answer}\n"
        f"Student response: {response}\n\n"
        f"{PARAPHRASE_PROMPTS[style]}"
    )
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        max_tokens=300,
    )
    return result.choices[0].message.content.strip()

def apply_llm_paraphrase(df_subset, style, aug_col='paraphrased'):
    """Paraphrase each response with an LLM. Returns a new df with binary aug flags."""
    new_rows = []
    for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Paraphrasing ({style})"):
        try:
            paraphrased = paraphrase_response(
                response=row['response'],
                passage=row['passage'],
                question=row['question'],
                answer=row['answer'],
                style=style,
            )
            new_row = row.copy()
            new_row['response'] = paraphrased
            new_rows.append(new_row)
        except Exception as e:
            print(f"  Skipped row (index={_}): {e}")
    result = pd.DataFrame(new_rows)
    return _set_aug_flags(result, aug_col)

## Build Augmented Dataset

Helpers:
- `apply_surface_aug` — 1 new row per original row (typos, caps, punct, numeric)
- `apply_multi_aug` — n new rows per original row (date formats)

LLM-based paraphrases added after this particular code block in a separate cell.


In [48]:
def apply_surface_aug(df_subset, transform_fn, aug_col, batch=True):
    """1 new row per input row. transform_fn takes a list and returns a list."""
    subset = df_subset.copy()
    if batch:
        subset['response'] = transform_fn(subset['response'].tolist())
    else:
        subset['response'] = subset['response'].apply(transform_fn)
    return _set_aug_flags(subset, aug_col)
    
def apply_multi_aug(df_subset, transform_fn, aug_col, n=3):
    """n new rows per input row. transform_fn(text, n) returns a list of variants."""
    new_rows = []
    for _, row in df_subset.iterrows():
        try:
            variants = transform_fn(row['response'], n=n)
            for v in variants:
                new_row = row.copy()
                new_row['response'] = v
                new_rows.append(new_row)
        except Exception:
            pass
    result = pd.DataFrame(new_rows)
    return _set_aug_flags(result, aug_col)


augmented_dfs = []

# 1. Date format variants (3 new rows per date row)
print("Augmenting dates...")
date_aug = apply_multi_aug(df[date_mask], augment_date, 'date_reformatted', n=3)
augmented_dfs.append(date_aug)
print(f" → {len(date_aug)} new rows")

# 2. Numeric → spelled-out (1 new row per numeric row)
print("Augmenting numerics...")
numeric_aug = apply_surface_aug(df[numeric_mask], augment_numeric, 'numeric_words', batch=False)
augmented_dfs.append(numeric_aug)
print(f" → {len(numeric_aug)} new rows")

# 3. Typos (all rows)
print("Augmenting typos...")
typo_aug = apply_surface_aug(df, augment_typos, 'misspelled', batch=True)
augmented_dfs.append(typo_aug)
print(f" → {len(typo_aug)} new rows")

# 4. Capitalization (all rows)
print("Augmenting capitalization...")
cap_aug = apply_surface_aug(df, augment_capitalization, 'capitalization_change', batch=True)
augmented_dfs.append(cap_aug)
print(f" → {len(cap_aug)} new rows")

# 5. Punctuation (all rows)
print("Augmenting punctuation...")
punct_aug = apply_surface_aug(df, augment_punctuation, 'punctuation_change', batch=True)
augmented_dfs.append(punct_aug)
print(f" → {len(punct_aug)} new rows")

# Original rows — all flags 0
df_original = df.copy()
for col in AUG_COLS:
    df_original[col] = 0

df_aug = pd.concat([df_original] + augmented_dfs, ignore_index=True)

print(f"\nOriginal rows: {len(df):>6}")
print(f"Total after augmentation: {len(df_aug):>6}")
print(f"\nAugmentation flag totals:")
print(df_aug[AUG_COLS].sum().to_string())

Augmenting dates...
 → 243 new rows
Augmenting numerics...
 → 1678 new rows
Augmenting typos...
 → 32076 new rows
Augmenting capitalization...
 → 32076 new rows
Augmenting punctuation...
 → 32076 new rows

Original rows:  32076
Total after augmentation: 130225

Augmentation flag totals:
misspelled               32076
paraphrased                  0
capitalization_change    32076
punctuation_change       32076
date_reformatted           243
numeric_words             1678


## LLM Paraphrases

Run these cells (the ones with LLM calls) separately. 
1. Shorter paraphrasing for score = 4
2. Longer parapharsing for score = 2
3. Combine/append these dataframes into df_aug

In [8]:
# Score = 4 → shorter paraphrases (1554 rows)
score4_aug = apply_llm_paraphrase(df[score4_mask], style='shorter', aug_col='paraphrased')
print(f"score=4 paraphrases: {len(score4_aug)} new rows")

Paraphrasing (shorter): 100%|██████████| 1554/1554 [19:08<00:00,  1.35it/s]


score=4 paraphrases: 1554 new rows


Paraphrasing (longer):   0%|          | 12/5024 [00:19<2:12:31,  1.59s/it]


KeyboardInterrupt: 

In [11]:
# Save score 4 augmentations
score4_aug.to_csv("score4_aug.csv")

In [ ]:
# NOTE: did not run this cell or the cell right after. 

# Score = 2 → longer paraphrases (5024 rows — sample to control cost)
# score2_sample = df[score2_mask].sample(n=2000, random_state=1)
score2_aug = apply_llm_paraphrase(df[score2_mask], style='longer', aug_col='paraphrased')
print(f"score=2 paraphrases: {len(score2_aug)} new rows")

score2_aug.to_csv("score2_aug.csv")

In [ ]:
# Append to df_aug
df_aug = pd.concat([df_aug, score4_aug, score2_aug], ignore_index=True)
print(f"\nTotal rows including LLM paraphrases: {len(df_aug)}")
print(f"\nAugmentation flag totals:")
print(df_aug[AUG_COLS].sum().to_string())

## Preview Augmented Data (w/out paraphrases)

Side-by-side comparison of original vs augmented response for each flag type.

In [26]:
# answer_id is unique per student response — use it to look up the original
original_lookup = df.set_index(['provided_split', 'answer_id'])['response'].to_dict()

def preview_aug(aug_col, n=5):
    """Show n side-by-side original/augmented pairs for a given flag column."""
    sample = df_aug[df_aug[aug_col] == 1].head(n)
    rows = []
    for _, row in sample.iterrows():
        rows.append({
            'original': original_lookup.get((row['provided_split'], row['answer_id']), '[not found]'),
            'augmented': row['response'],
            'answer':    row['answer'],
            'gpt5_score': int(row['gpt5_score']),
            'label':      int(row['label']),
        })
    return pd.DataFrame(rows)

# --- Per-flag previews ---
for col in AUG_COLS:
    print(f"\n{'='*60}")
    print(f"  {col}  ({int(df_aug[col].sum())} rows)")
    print('='*60)
    display(preview_aug(col, n=20))


  misspelled  (32076 rows)


,original,augmented,answer,gpt5_score,label
0,February 17,Feburary 17,She was shot on February 29.,1,0
1,February 29,February 2o,She was shot on February 29.,3,1
2,October 29,Octobwr 29,She was shot on February 29.,1,0
3,October 17,October 18,She was shot on February 29.,1,0
4,February 17,February 1&,She was shot on February 29.,1,0
5,Charleton Heston,Charleton Hestln,Charlton Heston,3,1
6,Moore,Moore,Charlton Heston,1,0
7,George Hoya,George Ho^a,Charlton Heston,1,0
8,Rolland,Rooland,Charlton Heston,1,0
9,Hoya,Hoys,Charlton Heston,1,0



  paraphrased  (0 rows)


""



  capitalization_change  (32076 rows)


...

,original,augmented,answer,gpt5_score,label
0,February 17,Febru...ary 1:7,She was shot on February 29.,1,0
1,February 29,Febru-ary 2;9,She was shot on February 29.,3,1
2,October 29,Octob.er 29,She was shot on February 29.,1,0
3,October 17,Octob:er 17,She was shot on February 29.,1,0
4,February 17,"Febru,ary 1.7",She was shot on February 29.,1,0
5,Charleton Heston,"Charl,eton ;Hesto?n",Charlton Heston,3,1
6,Moore,Moore,Charlton Heston,1,0
7,George Hoya,Georg;e Hoy!a,Charlton Heston,1,0
8,Rolland,Rolla'nd,Charlton Heston,1,0
9,Hoya,Hoya,Charlton Heston,1,0



  date_reformatted  (243 rows)


,original,augmented,answer,gpt5_score,label
0,February 17,"Feb. 17th, 2026",She was shot on February 29.,1,0
1,February 17,"Feb. 17, 2026",She was shot on February 29.,1,0
2,February 17,2/17/2026,She was shot on February 29.,1,0
3,October 29,October 29,She was shot on February 29.,1,0
4,October 29,October 29th,She was shot on February 29.,1,0
5,October 29,Oct. 29,She was shot on February 29.,1,0
6,October 17,"Oct. 17, 2026",She was shot on February 29.,1,0
7,October 17,2026-10-17,She was shot on February 29.,1,0
8,October 17,"October 17, 2026",She was shot on February 29.,1,0
9,February 17,"February 17, 2026",She was shot on February 29.,1,0



  numeric_words  (1678 rows)


,original,augmented,answer,gpt5_score,label
0,2,Two,Once.,1,1
1,3,Three,Once.,1,0
2,10 days,Ten days,He needed replacement parts — specifically a t...,1,0
3,5 years,Five years,22 years (1905 − 1883 = 22),1,0
4,22 years had passed,Twenty-two years had passed,22 years (1905 − 1883 = 22),4,1
5,20 years had passed,Twenty years had passed,22 years (1905 − 1883 = 22),1,0
6,22 years,Twenty-two years,22 years (1905 − 1883 = 22),3,1
7,22 years from 1883 to 1905,Twenty-two years from 1883 to 1905,22 years (1905 − 1883 = 22),4,1
8,1,One,Three. The people against the stranger are Jas...,1,0
9,2,Two,Three. The people against the stranger are Jas...,2,1


# Preview Paraphrases

Shown below is only shorter paraphrases for score = 4. 

- original: student response
- response: LLM-augmented answer
- answer: reference answer

In [42]:
original_lookup = df.set_index(['provided_split', 'answer_id'])['response'].to_dict()

preview = score4_aug.head(10).copy()
preview['original'] = preview.apply(lambda r: original_lookup.get((r['provided_split'], r['answer_id']), '[not found]'), axis=1)
preview[['original', 'response', 'answer', 'gpt5_score', 'label']]

,original,response,answer,gpt5_score,label
train,A round tire,A tire.,The tire.,4,1
train,Tire and sun gatherer,Tire and sun gatherer.,He was missing a tire and a sun gatherer.,4,1
train,Two parts namely tire and sun gatherer,Tire and sun gatherer.,He was missing a tire and a sun gatherer.,4,1
train,22 years had passed,22 years passed.,22 years (1905 − 1883 = 22),4,1
train,22 years from 1883 to 1905,22 years from 1883 to 1905.,22 years (1905 − 1883 = 22),4,1
train,Alexander II's death prevented the plans for a Duma to come to fruition,Alexander II's death stopped the Duma plans.,Alexander II's assassination (his death) prevented his plans for a Duma from coming to fruition.,4,1
train,Calaveras Kate was relieved because the stranger had his pistol out first and didn't pull the tr...,Calaveras Kate is relieved because the stranger drew first but didn't shoot.,"Calaveras Kate is relieved because the stranger refuses to pull the trigger, so Jason (whom she ...",4,1
train,"The suspects are Rafael Moreno, Jason Carberry, and Billy Buckett","The suspects are Rafael Moreno, Jason Carberry, and Billy Buckett.","Rafael Moreno, Jason Carberry, or Billy Buckett.",4,1
train,The stranger believes one of their men killed his brother. They are waiting to see who will survive,"The stranger thinks one of their men killed his brother, and they're anxious about who will surv...",Because there is violent conflict and a possible shootout—Reb Randall is hunting his brother’s k...,4,1
train,"Reb Randall, the strange man in the town","Reb Randall, the stranger in town.",Reb Randall,4,1
